# 28. Biomechanical Proxy Test (Pipeline 09)

- Goal: inspect relative CoM, moment-arm, and load-shift proxy records as a separate biomechanical proxy layer.
- Docs: `docs_eng/pipeline/09_biomechanical_proxy.md` / `docs/pipeline/09_biomechanical_proxy.md`
- Prerequisite: 20-27 stage checks should already pass.
- Inputs: ①-⑧ prepared p01 `squat` dataframe; downstream coordinates remain `norm`.
- Outputs: In-memory `BiomechRecord` list and pipeline biomech report.
- Checks: Biomech record counts, relative units, source-field provenance, and pipeline integration.

---


In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import re

import pandas as pd

from movement.biomech import (
    BIOMECH_REQUIRED_COLUMNS,
    BiomechRecord,
    extract_rep_biomech,
    save_biomech_outputs,
)
from movement.canonicalization import CanonicalizationConfig
from movement.config import LANDMARKS
from movement.exercise_definition import load_exercise_definition
from movement.io import load_pose_csv
from movement.pipeline import NormalizationConfig, PreprocessingConfig, run_pipeline
from movement.stage_context import (
    build_stage_check_pipeline_config,
    find_project_root,
    recording_id_from_pose_csv,
    resolve_target_definitions_dir,
)
from movement.stages.annotation import load_annotation_csv

print('imports OK')


## Data Setup

Run the selected sample through ①-⑧ with the current `squat` definition. ⑥ canonicalization is report-only and keeps downstream biomech computation on `norm` coordinates.


In [ ]:
PROJECT_ROOT = find_project_root()

pose_csv = 'data/pose/mediapipe/no_consent/20260517/p01_squat_set1_output_pose.csv'
annotation_csv = 'data/pose/mediapipe/no_consent/20260517/p01_squat_set1_annotation.csv'
TARGET_EXERCISE_ID = 'squat'
RECORDING_ID = recording_id_from_pose_csv(pose_csv)
BIOMECH_OUTPUT_DIR = PROJECT_ROOT / 'data' / 'processed' / 'biomech'

TARGET_DEFINITIONS_DIR = resolve_target_definitions_dir(
    TARGET_EXERCISE_ID,
    project_root=PROJECT_ROOT,
)
df_raw = load_pose_csv(PROJECT_ROOT / pose_csv)
ann_df = load_annotation_csv(PROJECT_ROOT / annotation_csv)
exercise_def = load_exercise_definition(
    exercise_id=TARGET_EXERCISE_ID,
    definitions_dir=TARGET_DEFINITIONS_DIR,
)


def make_stage_config(*, enable_biomech=False):
    canonical_config = CanonicalizationConfig(
        enabled=True,
        report_only=True,
        downstream_coordinate_mode='norm',
    )
    return build_stage_check_pipeline_config(
        exercise_id=TARGET_EXERCISE_ID,
        definitions_dir=TARGET_DEFINITIONS_DIR,
        annotation_csv=annotation_csv,
        preprocessing_config=PreprocessingConfig(enabled=True),
        normalization_config=NormalizationConfig(
            enabled=True,
            keep_reference_columns=True,
            model_depth_scale=1.0,
        ),
        canonicalization_config=canonical_config,
        enable_canonicalization=True,
        enable_rep_segmentation=True,
        enable_phase_segmentation=True,
        enable_features=True,
        enable_role_context=True,
        enable_biomech=enable_biomech,
    )


upstream_df, upstream_report = run_pipeline(
    df_raw,
    config=make_stage_config(enable_biomech=False),
    landmarks=LANDMARKS,
    ann_df=ann_df,
)

setup_summary = pd.DataFrame([
    {'item': 'recording_id', 'value': RECORDING_ID},
    {'item': 'exercise_id', 'value': exercise_def.exercise_id},
    {'item': 'definitions_dir', 'value': str(TARGET_DEFINITIONS_DIR.relative_to(PROJECT_ROOT))},
    {'item': 'frames_loaded', 'value': len(df_raw)},
    {'item': 'rep_count', 'value': int(upstream_df.loc[upstream_df['segment_type'].eq('rep'), 'rep_id'].dropna().nunique())},
    {'item': 'phase_labels', 'value': ', '.join(map(str, sorted(upstream_df['phase'].dropna().unique())))},
    {'item': 'upstream_steps', 'value': ', '.join(upstream_report.keys())},
    {'item': 'canonicalization_downstream_mode', 'value': upstream_report.get('canonicalization', {}).get('downstream_coordinate_mode')},
    {'item': 'feature_records', 'value': len(upstream_report.get('features', []))},
])
display(setup_summary)


## Direct extract_rep_biomech() Test


In [ ]:
records = extract_rep_biomech(upstream_df, exercise_def, use_visibility_weight=True)
biomech_df = pd.DataFrame([record.__dict__ for record in records])
biomech_df['metric_family'] = biomech_df['metric_id'].str.split('.').str[:2].str.join('.')

metric_summary = (
    biomech_df.groupby(['metric_family', 'unit'], dropna=False)
    .agg(records=('metric_id', 'size'), reps=('rep_id', lambda s: int(pd.Series(s).dropna().nunique())))
    .reset_index()
)
display(metric_summary)
print(f'BiomechRecord count: {len(records)}')


## Check 1: BiomechRecord Fields

In [ ]:
required_cols = BIOMECH_REQUIRED_COLUMNS

sequence_level_records = []
for record in records:
    assert isinstance(record, BiomechRecord)
    assert record.exercise_id == TARGET_EXERCISE_ID
    assert record.metric_id, 'metric_id empty'
    assert record.value is not None, f'value None for {record.metric_id}'
    assert record.unit, 'unit empty'
    assert record.source_fields, f'source_fields empty for {record.metric_id}'
    assert record.availability in {'assessed', 'low_confidence', 'not_assessed'}
    assert record.depth_dependency in {'none', 'low', 'moderate', 'high', 'unknown'}
    assert record.model_depth_reliability in {'high', 'moderate', 'low', 'unknown'}
    if record.rep_id is None:
        sequence_level_records.append(record)

for col in required_cols:
    assert col in biomech_df.columns, f'missing biomech column: {col}'

print(f'PASS: all {len(records)} BiomechRecord fields valid')
print(f'sequence-level biomech records with rep_id=None: {len(sequence_level_records)}')


## Check 2: No Absolute Units

In [ ]:
forbidden = re.compile(r'\b(N|Nm|N\.m|kg|meter|newton)\b', re.IGNORECASE)
for record in records:
    assert not forbidden.search(record.unit), f'absolute unit in {record.metric_id}: {record.unit}'
print('PASS: no absolute units; all biomech outputs remain relative proxies')
print(f'units found: {set(record.unit for record in records)}')


## Check 3: CoM and Moment Arm Records Present


In [ ]:
metric_ids = [record.metric_id for record in records]
has_com = any('com' in metric_id for metric_id in metric_ids)
has_moment_arm = any('moment_arm' in metric_id for metric_id in metric_ids)
has_load_shift = any('load_shift' in metric_id for metric_id in metric_ids)

display(
    biomech_df.assign(metric_group=biomech_df['metric_id'].str.extract(r'^(biomech\.[^.]+)')[0])
    .groupby(['metric_group', 'rep_id'], dropna=False)
    .size()
    .rename('records')
    .reset_index()
    .head(20)
)
assert has_com, 'no CoM metrics found'
assert has_moment_arm, 'no moment arm metrics found'
assert has_load_shift, 'no load-shift metrics found'
print('PASS: CoM, moment-arm, and load-shift proxy records are present')


## Check 4: Visibility Weighting Applied

In [ ]:
for record in records:
    assert record.visibility_weight_applied is not None, f'visibility_weight_applied missing for {record.metric_id}'
    assert record.n_frames_used is not None, 'n_frames_used missing'
    assert record.n_frames_used > 0, f'n_frames_used == 0 for {record.metric_id}'

visibility_summary = (
    biomech_df.groupby('metric_family', dropna=False)
    .agg(
        min_frames_used=('n_frames_used', 'min'),
        max_excluded_low_visibility=('n_frames_excluded_low_visibility', 'max'),
    )
    .reset_index()
)
display(visibility_summary)
print('PASS: visibility weighting fields present and valid')


## Check 5: Saved Output Contract

Save the ⑨ biomech proxy table and compact QC report, then reload the CSV to verify that later stage checks can use the file contract.


In [ ]:
saved_summary = save_biomech_outputs(
    biomech_df=biomech_df,
    recording_id=RECORDING_ID,
    exercise_id=exercise_def.exercise_id,
    output_dir=BIOMECH_OUTPUT_DIR,
    project_root=PROJECT_ROOT,
    required_columns=required_cols,
)
display(saved_summary)
print('PASS: saved Biomechanical Proxy outputs round-trip with the required contract')


## Check 6: Pipeline Integration


In [ ]:
pipe_df, pipe_report = run_pipeline(
    df_raw,
    config=make_stage_config(enable_biomech=True),
    landmarks=LANDMARKS,
    ann_df=ann_df,
)

assert 'biomech' in pipe_report
pipe_biomech = pd.DataFrame(pipe_report['biomech'])
assert len(pipe_biomech) == len(records)
assert set(pipe_biomech['metric_id']) == set(biomech_df['metric_id'])

pipeline_summary = pd.DataFrame([
    {'item': 'pipeline_biomech_records', 'value': len(pipe_biomech)},
    {'item': 'direct_biomech_records', 'value': len(records)},
    {'item': 'pipeline_feature_records', 'value': len(pipe_report.get('features', []))},
    {'item': 'steps_executed', 'value': ', '.join(pipe_report.keys())},
])
display(pipeline_summary)
print('PASS: pipeline ⑨ biomech report matches direct extraction on the ①-⑧ dataframe')


## Check Summary

This notebook is the follow-along checkpoint for ⑨ Biomechanical Proxy. It uses the current p01 `squat` flow through ①-⑧ first, then checks ⑨ as a separate relative-proxy layer. Biomech values remain low-confidence relative evidence by default and are not absolute force, torque, mass, or clinical diagnosis. The notebook saves CSV/QC artifacts under `data/processed/biomech/` so later stage checks can verify the file contract.
